#Importação de Bibliotecas

In [1]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras import layers
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Carregamento dos dados

In [2]:
from zipfile import ZipFile

with ZipFile('archive.zip', 'r') as zip:
    zip.extractall()
    print('Done')

Done


In [3]:
SEED = 42
EPOCHS  = 1000
learning_rate = 0.001

In [4]:
root = "/content/FINAL FOOD DATASET"
groups = []

for i in range(1, 6):
    filename = f"FOOD-DATA-GROUP{i}.csv"
    filepath = os.path.join(root, filename)
    df = pd.read_csv(filepath).drop(["Unnamed: 0", "Unnamed: 0.1"], axis=1)
    groups.append(df)

# Agora você tem todos os grupos armazenados na lista 'groups'
group1, group2, group3, group4, group5 = groups

data = pd.concat(groups)
data.head()

,food,Caloric Value,Fat,Saturated Fats,Monounsaturated Fats,Polyunsaturated Fats,Carbohydrates,Sugars,Protein,Dietary Fiber,...,Calcium,Copper,Iron,Magnesium,Manganese,Phosphorus,Potassium,Selenium,Zinc,Nutrition Density
0,cream cheese,51,5.0,2.9,1.3,0.200,0.8,0.500,0.9,0.0,...,0.008,14.100,0.082,0.027,1.300,0.091,15.5,19.100,0.039,7.070
1,neufchatel cheese,215,19.4,10.9,4.9,0.800,3.1,2.700,7.8,0.0,...,99.500,0.034,0.100,8.500,0.088,117.300,129.2,0.054,0.700,130.100
2,requeijao cremoso light catupiry,49,3.6,2.3,0.9,0.000,0.9,3.400,0.8,0.1,...,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,5.400
3,ricotta cheese,30,2.0,1.3,0.5,0.002,1.5,0.091,1.5,0.0,...,0.097,41.200,0.097,0.096,4.000,0.024,30.8,43.800,0.035,5.196
4,cream cheese low fat,30,2.3,1.4,0.6,0.042,1.2,0.900,1.2,0.0,...,22.200,0.072,0.008,1.200,0.098,22.800,37.1,0.034,0.053,27.007


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2395 entries, 0 to 721
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   food                  2395 non-null   object 
 1   Caloric Value         2395 non-null   int64  
 2   Fat                   2395 non-null   float64
 3   Saturated Fats        2395 non-null   float64
 4   Monounsaturated Fats  2395 non-null   float64
 5   Polyunsaturated Fats  2395 non-null   float64
 6   Carbohydrates         2395 non-null   float64
 7   Sugars                2395 non-null   float64
 8   Protein               2395 non-null   float64
 9   Dietary Fiber         2395 non-null   float64
 10  Cholesterol           2395 non-null   float64
 11  Sodium                2395 non-null   float64
 12  Water                 2395 non-null   float64
 13  Vitamin A             2395 non-null   float64
 14  Vitamin B1            2395 non-null   float64
 15  Vitamin B11           2395 

In [6]:
unics, counts = np.unique(data["food"], return_counts=True)

unics, counts

(array(['abalone', 'abiyuch', 'acerola cherry', ..., 'zucchini cooked',
        'zucchini raw', 'zwieback'], dtype=object),
 array([1, 1, 1, ..., 1, 1, 1]))

# Análise do Padrão Nutricional

In [7]:
def plot_nutrient(food: str, data: pd.DataFrame=data):
    # Filtrar o dataframe para o alimento especificado
    if food not in data['food'].values:
        print(f"O alimento '{food}' não foi encontrado.")
        return

    # Filtra a linha do alimento específico
    nutrient_data = data[data['food'] == food].iloc[0]

    # Remove a coluna com o nome do alimento para focar nos nutrientes
    nutrients = nutrient_data.drop(labels=['food'])

    # Transformar os dados em um formato apropriado para o Seaborn
    nutrients_df = nutrients.reset_index()
    nutrients_df.columns = ['Nutrient', 'Quantity']

    # Plotar os nutrientes usando Seaborn
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Nutrient', y='Quantity', hue='Nutrient', data=nutrients_df, palette='pastel', dodge=False)

    # Remover a legenda automática já que não precisamos dela
    plt.legend([],[], frameon=False)

    # Adicionar título e rótulos aos eixos
    plt.title(f"Nutrientes de {food}")
    plt.xlabel("Nutrientes")
    plt.ylabel("Quantidade")
    plt.xticks(rotation=75)

    # Mostrar o gráfico
    plt.show()

In [ ]:
for _ in range(5):
  random_food = data.iloc[np.random.randint(len(data))]['food']
  plot_nutrient(random_food)
  print()

In [9]:
def search_foods(name_nutrient: str, data: pd.DataFrame = data, ascending: bool = False, n: int = 10):
    # Verificar se o nutriente especificado existe no dataframe
    if name_nutrient not in data.columns:
        print(f"Nutriente '{name_nutrient}' não encontrado no conjunto de dados.")
        return

    # Ordenar os alimentos com base no nutriente especificado
    foods = data.sort_values(by=name_nutrient, ascending=ascending)[['food', name_nutrient]].head(n)

    # Exibir mensagem indicando a ordem
    order = "menores" if ascending else "maiores"
    print(f"Os {n} alimentos com os {order} valores de {name_nutrient}:")

    print(foods)

In [10]:
# Example 1
search_foods('Sugars')

Os 10 alimentos com os maiores valores de Sugars:
                          food  Sugars
71            vanilla frosting   291.5
25       danone low fat alsafi   255.0
156           banana cream pie   143.0
295  cranberry sauce sweetened   105.0
672        sweet potato canned    98.3
110             golden raisins    97.7
112       zante currants dried    96.9
195        chocolate semisweet    91.6
68         saccharin sweetener    85.2
81                almond paste    82.3


In [11]:
# Example 2
search_foods('Vitamin A')

Os 10 alimentos com os maiores valores de Vitamin A:
                          food  Vitamin A
43            dessert wine dry      362.7
221      hyacinth beans cooked      134.1
24   muller thurgau white wine      129.2
19              gamay red wine      128.6
42         semillon white wine      128.1
51       fume blanc white wine      127.9
35          lemberger red wine      127.9
49             claret red wine      127.0
29       petite sirah red wine      126.7
25           mouvedre red wine      126.1


In [12]:
# Example 3
search_foods('Potassium')

Os 10 alimentos com os maiores valores de Potassium:
                         food  Potassium
504       pork arm picnic raw    11336.9
506    pork arm picnic cooked     8039.6
469  pork top loin roasts raw     7858.1
536         turkey breast raw     6226.0
567            goose meat raw     5045.0
568     turkey breast roasted     4976.6
443     oriental radish dried     4053.0
252     potato chips fat free     3695.6
230           white beans raw     3625.9
49      instant coffee powder     3535.0


In [ ]:
# ALL COLUMNS
for col in data.columns:
  if col != "food":
    search_foods(col)
    print("-"*65)

In [14]:
correlation_check = data.drop("food", axis=1).corr()
correlations = correlation_check['Nutrition Density'].sort_values(ascending=False)

for column, correlation in correlations.items():
  print(f"{column}: {correlation:.3f}")

Nutrition Density: 1.000
Calcium: 0.796
Phosphorus: 0.558
Caloric Value: 0.535
Potassium: 0.529
Vitamin C: 0.491
Magnesium: 0.471
Protein: 0.455
Fat: 0.422
Vitamin B3: 0.393
Water: 0.359
Monounsaturated Fats: 0.358
Zinc: 0.332
Iron: 0.325
Carbohydrates: 0.323
Vitamin B6: 0.303
Polyunsaturated Fats: 0.285
Dietary Fiber: 0.274
Vitamin B1: 0.233
Vitamin B2: 0.210
Vitamin B5: 0.209
Saturated Fats: 0.202
Vitamin E: 0.182
Sodium: 0.169
Cholesterol: 0.145
Sugars: 0.115
Vitamin A: 0.046
Vitamin B11: 0.003
Vitamin B12: -0.006
Vitamin K: -0.017
Copper: -0.022
Manganese: -0.034
Selenium: -0.051
Vitamin D: -0.080


# Modelagem preditiva para impactos na saúde

In [15]:
data.drop("food", axis=1, inplace=True)
train_stats = data.describe()
train_stats = train_stats.transpose()

In [16]:
def data_normalizer(x):
  return (x - train_stats['mean']) / train_stats['std']

data = data_normalizer(data)

In [17]:
X = data.drop("Nutrition Density", axis=1).values
y = data["Nutrition Density"].values.reshape(-1, 1)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [19]:
class Model(keras.Model):
  def __init__(self):
    super(Model, self).__init__()
    self.dense1 = layers.Dense(64, activation='relu')
    self.dense2 = layers.Dense(64, activation='relu')
    self.dense3 = layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

In [20]:
class PrintDot(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs):
        if epoch % 100 == 0: print('')
        print('.', end='')
        if epoch == EPOCHS - 1 or epoch == logs.get('val_loss', None):
            print('')

In [21]:
def plot_metrics(history):
  hist = pd.DataFrame(history.history)
  hist['epoch'] = history.epoch

  print(hist.tail(10))

In [22]:
def plot_errors(history):
  hist = pd.DataFrame(history.history)
  hist['epoch'] = history.epoch

  plt.figure()
  plt.xlabel('Epoch')
  plt.ylabel('Mean Abs Error [GPA]')
  plt.plot(hist['epoch'], hist['mae'], label='Train Error')
  plt.plot(hist['epoch'], hist['val_mae'], label='Val Error')
  plt.ylim([0, hist['val_mae'].max()*1.1])
  plt.legend()

  plt.figure()
  plt.xlabel('Epoch')
  plt.ylabel('Mean Square Error [$GPA^2$]')
  plt.plot(hist['epoch'], hist['mse'], label='Train Error')
  plt.plot(hist['epoch'], hist['val_mse'], label='Val Error')
  plt.ylim([0, hist['val_mse'].max()*1.1])
  plt.legend()
  plt.show()

In [23]:
model = Model()
model.compile(loss = keras.losses.Huber(reduction=tf.keras.losses.Reduction.SUM),
              optimizer=keras.optimizers.Adam(learning_rate),
              metrics=['mae', 'mse'])

In [24]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)

history = model.fit(X_train, y_train, epochs=EPOCHS,
                    validation_data = (X_test, y_test), verbose=0, callbacks=[early_stop, PrintDot()])


..........................

In [25]:
plot_metrics(history)

        loss       mae       mse  val_loss   val_mae   val_mse  epoch
16  0.157199  0.042775  0.010997  0.216577  0.051313  0.013658     16
17  0.210664  0.045636  0.015652  0.294198  0.061766  0.018480     17
18  0.209755  0.051365  0.017229  0.235947  0.054162  0.015503     18
19  0.202958  0.047277  0.016501  0.315540  0.076630  0.019822     19
20  0.254598  0.053635  0.022856  0.266548  0.061295  0.017368     20
21  0.141708  0.041132  0.011595  0.308028  0.054560  0.020130     21
22  0.102085  0.037643  0.006649  0.234107  0.053587  0.015524     22
23  0.210470  0.048213  0.016164  0.425820  0.075790  0.027471     23
24  0.216988  0.057374  0.014714  0.305806  0.066180  0.021378     24
25  0.174815  0.049625  0.012217  0.344118  0.061434  0.022640     25


In [ ]:
plot_errors(history)